In [0]:
%sql
CREATE VOLUME databricks_de_project.default.raw_volume;

In [0]:
display(
    dbutils.fs.ls("/Volumes/databricks_de_project/default/raw_volume/")
)

path,name,size,modificationTime
dbfs:/Volumes/databricks_de_project/default/raw_volume/customers.csv,customers.csv,5637,1777191184000
dbfs:/Volumes/databricks_de_project/default/raw_volume/orders.csv,orders.csv,2333,1777191184000
dbfs:/Volumes/databricks_de_project/default/raw_volume/products.csv,products.csv,3566,1777191184000


In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .load("dbfs:/Volumes/databricks_de_project/default/raw_volume/customers.csv")

display(df)

customer_id,first_name,last_name,email,country
1,Amy,Nelson,amy.nelson85@gmail.com,Algeria
2,Kristen,Hill,kristen.hill26@gmail.com,Nepal
3,Abigail,Martin,abigail.martin35@hotmail.com,Liberia
4,Tiffany,Murray,tiffany.murray46@gmail.com,El Salvador
5,Sarah,Williams,sarah.williams40@yahoo.com,Italy
6,Julie,Drake,julie.drake65@gmail.com,Somalia
7,Neil,Adams,neil.adams21@hotmail.com,Tunisia
8,Sandra,Powers,sandra.powers79@hotmail.com,Sudan
9,Stacey,Archer,stacey.archer11@gmail.com,Dominica
10,Jack,White,jack.white86@hotmail.com,Mayotte


In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .load("dbfs:/Volumes/databricks_de_project/default/raw_volume/orders.csv")

display(df)

order_id,customer_id,product_id,quantity,order_date
1,22,75,1,2026-01-15
2,89,24,4,2026-01-27
3,13,59,4,2026-03-13
4,26,90,1,2025-11-08
5,81,29,1,2025-12-30
6,18,60,2,2026-02-19
7,14,72,1,2026-02-15
8,47,72,1,2026-04-13
9,73,79,3,2026-02-26
10,67,29,1,2025-12-03


In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .load("dbfs:/Volumes/databricks_de_project/default/raw_volume/products.csv")

display(df)

product_id,product_name,category,price
1,Nike Samsung Galaxy,Electronics,1444.07
2,HP Cookware Set,Home,1039.65
3,Lenovo Tennis Racket,Sports,834.93
4,Puma Jeans,Clothing,950.49
5,Samsung Notebook,Books,1015.74
6,Puma Headphones,Electronics,915.69
7,Puma iPhone,Electronics,1343.78
8,Lenovo Office Chair,Home,1252.53
9,Sony Yoga Mat,Sports,891.89
10,Apple Football,Sports,1044.61


🥉 Bronze Layer (Raw → Delta Tables)

Step 1: Create Bronze Tables

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS databricks_de_project.bronze;

Step 2: Ingest Customers

In [0]:
df_customers = spark.read.format("csv") \
    .option("header", "true") \
    .load("/Volumes/databricks_de_project/default/raw_volume/customers.csv")

df_customers.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("databricks_de_project.bronze.customers")

Step 3: Ingest Products

In [0]:
df_products = spark.read.format("csv") \
    .option("header", "true") \
    .load("/Volumes/databricks_de_project/default/raw_volume/products.csv")

df_products.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("databricks_de_project.bronze.products")

Step 4: Ingest Orders

In [0]:
df_orders = spark.read.format("csv") \
    .option("header", "true") \
    .load("/Volumes/databricks_de_project/default/raw_volume/orders.csv")

df_orders.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("databricks_de_project.bronze.orders")

Step 5: Verify Tables

In [0]:
%sql
SELECT * FROM databricks_de_project.bronze.customers;

customer_id,first_name,last_name,email,country
1,Amy,Nelson,amy.nelson85@gmail.com,Algeria
2,Kristen,Hill,kristen.hill26@gmail.com,Nepal
3,Abigail,Martin,abigail.martin35@hotmail.com,Liberia
4,Tiffany,Murray,tiffany.murray46@gmail.com,El Salvador
5,Sarah,Williams,sarah.williams40@yahoo.com,Italy
6,Julie,Drake,julie.drake65@gmail.com,Somalia
7,Neil,Adams,neil.adams21@hotmail.com,Tunisia
8,Sandra,Powers,sandra.powers79@hotmail.com,Sudan
9,Stacey,Archer,stacey.archer11@gmail.com,Dominica
10,Jack,White,jack.white86@hotmail.com,Mayotte


In [0]:
%sql
SELECT * FROM databricks_de_project.bronze.products;

product_id,product_name,category,price
1,Nike Samsung Galaxy,Electronics,1444.07
2,HP Cookware Set,Home,1039.65
3,Lenovo Tennis Racket,Sports,834.93
4,Puma Jeans,Clothing,950.49
5,Samsung Notebook,Books,1015.74
6,Puma Headphones,Electronics,915.69
7,Puma iPhone,Electronics,1343.78
8,Lenovo Office Chair,Home,1252.53
9,Sony Yoga Mat,Sports,891.89
10,Apple Football,Sports,1044.61


In [0]:
%sql
SELECT * FROM databricks_de_project.bronze.orders;

order_id,customer_id,product_id,quantity,order_date
1,22,75,1,2026-01-15
2,89,24,4,2026-01-27
3,13,59,4,2026-03-13
4,26,90,1,2025-11-08
5,81,29,1,2025-12-30
6,18,60,2,2026-02-19
7,14,72,1,2026-02-15
8,47,72,1,2026-04-13
9,73,79,3,2026-02-26
10,67,29,1,2025-12-03


Raw CSV → Bronze Delta Tables

Add schema explicitly (important for production):

In [0]:
from pyspark.sql.types import *

customer_schema = StructType([
    StructField("first_name", StringType()),
    StructField("last_name", StringType()),
    StructField("email", StringType()),
    StructField("country", StringType())
])

Verify Storage

In [0]:
%sql
DESCRIBE DETAIL databricks_de_project.bronze.customers;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,19ed10b5-c4dd-4e03-9026-6bda56d0cf45,databricks_de_project.bronze.customers,null,,2026-04-26T08:22:11.021Z,2026-04-26T08:22:14.000Z,List(),List(),1,4983,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
DESCRIBE DETAIL databricks_de_project.bronze.products;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,cada665b-1c2b-4494-8f48-36f8d86ea6b8,databricks_de_project.bronze.products,null,,2026-04-26T08:23:13.913Z,2026-04-26T08:23:15.000Z,List(),List(),1,2638,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
DESCRIBE DETAIL databricks_de_project.bronze.orders;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,a3aaa73d-08ff-4181-ae65-7d43b9af5b80,databricks_de_project.bronze.orders,null,,2026-04-26T08:24:09.634Z,2026-04-26T08:24:10.000Z,List(),List(),1,2576,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


🥈 Phase 3: Silver Layer (Transformations)

Bronze (raw, messy) → Silver (clean, enriched, usable)

Step 1: Create Silver Schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS databricks_de_project.silver;

Step 2: Customers Transformation

Customers
- Clean data
- Create full_name
- Extract email domain

🧠 What You Just Did

✔ Derived column (full_name)
✔ Feature extraction (email_domain)
✔ Deduplication

In [0]:
from pyspark.sql.functions import col, concat, lit, split

df = spark.table("databricks_de_project.bronze.customers")

df_silver = df \
    .withColumn("full_name", concat(col("first_name"), lit(" "), col("last_name"))) \
    .withColumn("email_domain", split(col("email"), "@")[1]) \
    .dropDuplicates()

df_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("databricks_de_project.silver.customers")

In [0]:
%sql
SELECT * FROM databricks_de_project.silver.customers;

customer_id,first_name,last_name,email,country,full_name,email_domain
2,Kristen,Hill,kristen.hill26@gmail.com,Nepal,Kristen Hill,gmail.com
10,Jack,White,jack.white86@hotmail.com,Mayotte,Jack White,hotmail.com
28,Robin,Martinez,robin.martinez8@gmail.com,Singapore,Robin Martinez,gmail.com
34,Madeline,Chen,madeline.chen8@yahoo.com,Vanuatu,Madeline Chen,yahoo.com
42,Benjamin,Skinner,benjamin.skinner47@yahoo.com,Vanuatu,Benjamin Skinner,yahoo.com
53,Stephen,Roman,stephen.roman35@yahoo.com,Macao,Stephen Roman,yahoo.com
55,Jason,Ford,jason.ford71@hotmail.com,French Guiana,Jason Ford,hotmail.com
82,Christopher,Martinez,christopher.martinez10@gmail.com,Egypt,Christopher Martinez,gmail.com
88,Rebecca,Howe,rebecca.howe27@yahoo.com,Moldova,Rebecca Howe,yahoo.com
3,Abigail,Martin,abigail.martin35@hotmail.com,Liberia,Abigail Martin,hotmail.com


Step 3: Products Transformation

Products
- Apply transformations
- Clean pricing

🧠 Why This Matters

✔ Enforcing schema (price as numeric)
✔ Avoids downstream errors

In [0]:
from pyspark.sql.functions import col

df = spark.table("databricks_de_project.bronze.products")

df_silver = df \
    .withColumn("price", col("price").cast("double"))

df_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("databricks_de_project.silver.products")

In [0]:
%sql
SELECT * FROM databricks_de_project.silver.products;

product_id,product_name,category,price
1,Nike Samsung Galaxy,Electronics,1444.07
2,HP Cookware Set,Home,1039.65
3,Lenovo Tennis Racket,Sports,834.93
4,Puma Jeans,Clothing,950.49
5,Samsung Notebook,Books,1015.74
6,Puma Headphones,Electronics,915.69
7,Puma iPhone,Electronics,1343.78
8,Lenovo Office Chair,Home,1252.53
9,Sony Yoga Mat,Sports,891.89
10,Apple Football,Sports,1044.61


Step 4: Orders Transformation

Orders
- Convert dates
- Add derived columns
- Prepare for joins

🧠 What You Did Here

✔ Converted string → date
✔ Extracted year (used later for analytics)

In [0]:
from pyspark.sql.functions import col, to_date, year

df = spark.table("databricks_de_project.bronze.orders")

df_silver = df \
    .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd")) \
    .withColumn("order_year", year(col("order_date")))

df_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("databricks_de_project.silver.orders")

In [0]:
%sql
SELECT * FROM databricks_de_project.silver.orders;

order_id,customer_id,product_id,quantity,order_date,order_year
1,22,75,1,2026-01-15,2026
2,89,24,4,2026-01-27,2026
3,13,59,4,2026-03-13,2026
4,26,90,1,2025-11-08,2025
5,81,29,1,2025-12-30,2025
6,18,60,2,2026-02-19,2026
7,14,72,1,2026-02-15,2026
8,47,72,1,2026-04-13,2026
9,73,79,3,2026-02-26,2026
10,67,29,1,2025-12-03,2025


🚀 Advanced Transformations

Customer Segmentation

Add:
- customer type based on email domain

In [0]:
from pyspark.sql.functions import when, col

df = spark.table("databricks_de_project.silver.customers")

df = df.withColumn(
    "customer_segment",
    when(col("email_domain") == "gmail.com", "GMAIL_USER")
    .when(col("email_domain") == "yahoo.com", "YAHOO_USER")
    .otherwise("OTHER")
)

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("databricks_de_project.silver.customers")

In [0]:
%sql
SELECT * FROM databricks_de_project.silver.customers;

customer_id,first_name,last_name,email,country,full_name,email_domain,customer_segment
2,Kristen,Hill,kristen.hill26@gmail.com,Nepal,Kristen Hill,gmail.com,GMAIL_USER
10,Jack,White,jack.white86@hotmail.com,Mayotte,Jack White,hotmail.com,OTHER
28,Robin,Martinez,robin.martinez8@gmail.com,Singapore,Robin Martinez,gmail.com,GMAIL_USER
34,Madeline,Chen,madeline.chen8@yahoo.com,Vanuatu,Madeline Chen,yahoo.com,YAHOO_USER
42,Benjamin,Skinner,benjamin.skinner47@yahoo.com,Vanuatu,Benjamin Skinner,yahoo.com,YAHOO_USER
53,Stephen,Roman,stephen.roman35@yahoo.com,Macao,Stephen Roman,yahoo.com,YAHOO_USER
55,Jason,Ford,jason.ford71@hotmail.com,French Guiana,Jason Ford,hotmail.com,OTHER
82,Christopher,Martinez,christopher.martinez10@gmail.com,Egypt,Christopher Martinez,gmail.com,GMAIL_USER
88,Rebecca,Howe,rebecca.howe27@yahoo.com,Moldova,Rebecca Howe,yahoo.com,YAHOO_USER
3,Abigail,Martin,abigail.martin35@hotmail.com,Liberia,Abigail Martin,hotmail.com,OTHER


In [0]:
%sql
SELECT customer_segment, COUNT(*) 
FROM databricks_de_project.silver.customers
GROUP BY customer_segment;

customer_segment,COUNT(*)
GMAIL_USER,42
OTHER,28
YAHOO_USER,30


Product Pricing Buckets

In [0]:
from pyspark.sql.functions import when

df = spark.table("databricks_de_project.silver.products")

df = df.withColumn(
    "price_category",
    when(col("price") < 300, "LOW")
    .when(col("price") < 800, "MEDIUM")
    .otherwise("HIGH")
)

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("databricks_de_project.silver.products")

In [0]:
%sql
SELECT * FROM databricks_de_project.silver.products;

product_id,product_name,category,price,price_category
1,Nike Samsung Galaxy,Electronics,1444.07,HIGH
2,HP Cookware Set,Home,1039.65,HIGH
3,Lenovo Tennis Racket,Sports,834.93,HIGH
4,Puma Jeans,Clothing,950.49,HIGH
5,Samsung Notebook,Books,1015.74,HIGH
6,Puma Headphones,Electronics,915.69,HIGH
7,Puma iPhone,Electronics,1343.78,HIGH
8,Lenovo Office Chair,Home,1252.53,HIGH
9,Sony Yoga Mat,Sports,891.89,HIGH
10,Apple Football,Sports,1044.61,HIGH


In [0]:
%sql
SELECT price_category, COUNT(*) 
FROM databricks_de_project.silver.products
GROUP BY price_category;

price_category,COUNT(*)
HIGH,57
LOW,16
MEDIUM,27


Order Value Calculation

In [0]:
df_orders = spark.table("databricks_de_project.silver.orders")
df_products = spark.table("databricks_de_project.silver.products")

df_joined = df_orders.join(df_products, "product_id", "left")

df_orders_enriched = df_joined.withColumn(
    "order_value",
    col("quantity") * col("price")
)

df_orders_enriched.write.mode("overwrite") \
    .saveAsTable("databricks_de_project.silver.orders_enriched")

In [0]:
%sql
SELECT * FROM databricks_de_project.silver.orders_enriched

product_id,order_id,customer_id,quantity,order_date,order_year,product_name,category,price,price_category,order_value
75,1,22,1,2026-01-15,2026,Puma Novel Book,Books,429.77,MEDIUM,429.77
24,2,89,4,2026-01-27,2026,Puma iPhone,Electronics,1134.68,HIGH,4538.72
59,3,13,4,2026-03-13,2026,HP Laptop,Electronics,888.49,HIGH,3553.96
90,4,26,1,2025-11-08,2025,Nike T-Shirt,Clothing,1432.59,HIGH,1432.59
29,5,81,1,2025-12-30,2025,Samsung Cookware Set,Home,519.57,MEDIUM,519.57
60,6,18,2,2026-02-19,2026,Lenovo Tennis Racket,Sports,715.95,MEDIUM,1431.9
72,7,14,1,2026-02-15,2026,Dell Sneakers,Clothing,1473.71,HIGH,1473.71
72,8,47,1,2026-04-13,2026,Dell Sneakers,Clothing,1473.71,HIGH,1473.71
79,9,73,3,2026-02-26,2026,Samsung Tennis Racket,Sports,105.14,LOW,315.42
29,10,67,1,2025-12-03,2025,Samsung Cookware Set,Home,519.57,MEDIUM,519.57


Customer Lifetime Value (CLV)

In [0]:
from pyspark.sql.functions import sum as _sum

df = spark.table("databricks_de_project.silver.orders_enriched")

df_clv = df.groupBy("customer_id") \
    .agg(_sum("order_value").alias("customer_lifetime_value"))

df_clv.write.mode("overwrite") \
    .saveAsTable("databricks_de_project.silver.customer_clv")

In [0]:
%sql
SELECT * FROM databricks_de_project.silver.orders_enriched;

product_id,order_id,customer_id,quantity,order_date,order_year,product_name,category,price,price_category,order_value
75,1,22,1,2026-01-15,2026,Puma Novel Book,Books,429.77,MEDIUM,429.77
24,2,89,4,2026-01-27,2026,Puma iPhone,Electronics,1134.68,HIGH,4538.72
59,3,13,4,2026-03-13,2026,HP Laptop,Electronics,888.49,HIGH,3553.96
90,4,26,1,2025-11-08,2025,Nike T-Shirt,Clothing,1432.59,HIGH,1432.59
29,5,81,1,2025-12-30,2025,Samsung Cookware Set,Home,519.57,MEDIUM,519.57
60,6,18,2,2026-02-19,2026,Lenovo Tennis Racket,Sports,715.95,MEDIUM,1431.9
72,7,14,1,2026-02-15,2026,Dell Sneakers,Clothing,1473.71,HIGH,1473.71
72,8,47,1,2026-04-13,2026,Dell Sneakers,Clothing,1473.71,HIGH,1473.71
79,9,73,3,2026-02-26,2026,Samsung Tennis Racket,Sports,105.14,LOW,315.42
29,10,67,1,2025-12-03,2025,Samsung Cookware Set,Home,519.57,MEDIUM,519.57


Top Customers (Window Function)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

window = Window.orderBy(col("customer_lifetime_value").desc())

df_ranked = df_clv.withColumn(
    "customer_rank",
    rank().over(window)
)

df_ranked.write.mode("overwrite") \
    .saveAsTable("databricks_de_project.silver.customer_ranking")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
%sql
SELECT * FROM databricks_de_project.silver.customer_ranking;

customer_id,customer_lifetime_value,customer_rank
58,7203.28,1
67,6921.209999999999,2
99,6223.98,3
73,6186.5599999999995,4
44,5820.780000000001,5
93,5618.950000000001,6
12,5344.23,7
4,5190.6900000000005,8
89,4538.72,9
52,4395.57,10


Time-Based Features

In [0]:
from pyspark.sql.functions import month

df = spark.table("databricks_de_project.silver.orders")

df = df.withColumn("order_month", month(col("order_date")))

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("databricks_de_project.silver.orders")

In [0]:
%sql
SELECT * FROM databricks_de_project.silver.orders;

order_id,customer_id,product_id,quantity,order_date,order_year,order_month
1,22,75,1,2026-01-15,2026,1
2,89,24,4,2026-01-27,2026,1
3,13,59,4,2026-03-13,2026,3
4,26,90,1,2025-11-08,2025,11
5,81,29,1,2025-12-30,2025,12
6,18,60,2,2026-02-19,2026,2
7,14,72,1,2026-02-15,2026,2
8,47,72,1,2026-04-13,2026,4
9,73,79,3,2026-02-26,2026,2
10,67,29,1,2025-12-03,2025,12


Data Quality Checks

In [0]:
df = spark.table("databricks_de_project.silver.customers")

df = df.filter(col("email").contains("@"))

df.write.mode("overwrite").saveAsTable("databricks_de_project.silver.customers")

In [0]:
%sql
SELECT * FROM databricks_de_project.silver.customers;

customer_id,first_name,last_name,email,country,full_name,email_domain,customer_segment
2,Kristen,Hill,kristen.hill26@gmail.com,Nepal,Kristen Hill,gmail.com,GMAIL_USER
10,Jack,White,jack.white86@hotmail.com,Mayotte,Jack White,hotmail.com,OTHER
28,Robin,Martinez,robin.martinez8@gmail.com,Singapore,Robin Martinez,gmail.com,GMAIL_USER
34,Madeline,Chen,madeline.chen8@yahoo.com,Vanuatu,Madeline Chen,yahoo.com,YAHOO_USER
42,Benjamin,Skinner,benjamin.skinner47@yahoo.com,Vanuatu,Benjamin Skinner,yahoo.com,YAHOO_USER
53,Stephen,Roman,stephen.roman35@yahoo.com,Macao,Stephen Roman,yahoo.com,YAHOO_USER
55,Jason,Ford,jason.ford71@hotmail.com,French Guiana,Jason Ford,hotmail.com,OTHER
82,Christopher,Martinez,christopher.martinez10@gmail.com,Egypt,Christopher Martinez,gmail.com,GMAIL_USER
88,Rebecca,Howe,rebecca.howe27@yahoo.com,Moldova,Rebecca Howe,yahoo.com,YAHOO_USER
3,Abigail,Martin,abigail.martin35@hotmail.com,Liberia,Abigail Martin,hotmail.com,OTHER


🥇 GOLD Layer

⭐ Dimension Tables
dim_customers
dim_products
⭐ Fact Table
fact_orders

Raw → Bronze → Silver → Gold
                    ↓
              Star Schema

🥇 GOLD LAYER — STAR SCHEMA

          dim_customers      dim_products
                 \             /
                  \           /
                   \         /
                   fact_orders

Step 1: Create Gold Schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS databricks_de_project.gold;

Step 2: Dim Customers (SCD Type-1 + Surrogate Key)

🧠 Goal:
One record per customer
Add surrogate key (used in fact table)

Deterministic Key

🧠 Why this is 🔥

✔ Stable key
✔ Reproducible
✔ Interview-level answer

In [0]:
from pyspark.sql.functions import col, sha2, concat_ws

df = spark.table("databricks_de_project.silver.customers")

df_dim = df.withColumn(
    "customer_key",
    sha2(concat_ws("||", col("customer_id"), col("email")), 256)
)

df_dim.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("databricks_de_project.gold.dim_customers")

In [0]:
%sql
SELECT * FROM databricks_de_project.gold.dim_customers

customer_id,first_name,last_name,email,country,full_name,email_domain,customer_segment,customer_key
2,Kristen,Hill,kristen.hill26@gmail.com,Nepal,Kristen Hill,gmail.com,GMAIL_USER,58d41f2a4959f103d408fff336c165ade812794e79f2194c0bf14f0642d9e794
10,Jack,White,jack.white86@hotmail.com,Mayotte,Jack White,hotmail.com,OTHER,00940bdf16b4a00f09d28d03612beaed97ecffb67e049643de6c8676c6c92e06
28,Robin,Martinez,robin.martinez8@gmail.com,Singapore,Robin Martinez,gmail.com,GMAIL_USER,65d8b2dde25b557bf6dedfc63fc6b5da947da75555acc7c2e4bdecdd8b34b7e6
34,Madeline,Chen,madeline.chen8@yahoo.com,Vanuatu,Madeline Chen,yahoo.com,YAHOO_USER,2ee239d77937a8955deb6cb3a526e8131d617a8b01cc1e22ac11bf18266d1e03
42,Benjamin,Skinner,benjamin.skinner47@yahoo.com,Vanuatu,Benjamin Skinner,yahoo.com,YAHOO_USER,d849e55866c6e1afb6037f4458e210a0bd3f5ba1df4c220f8d1a230f459d5ba0
53,Stephen,Roman,stephen.roman35@yahoo.com,Macao,Stephen Roman,yahoo.com,YAHOO_USER,594101e97535af6b7c228ab346d19d3d5cf6cf7822dd3e11719597cd7154dbf2
55,Jason,Ford,jason.ford71@hotmail.com,French Guiana,Jason Ford,hotmail.com,OTHER,57ce6d913726e160d86b6533334c7bf9d4bb15c7e89d2e4fc0f2efc3c0d00f43
82,Christopher,Martinez,christopher.martinez10@gmail.com,Egypt,Christopher Martinez,gmail.com,GMAIL_USER,42127bd523703c0978bda83f2687f653131cc59f4d198114b26a851dea83a63d
88,Rebecca,Howe,rebecca.howe27@yahoo.com,Moldova,Rebecca Howe,yahoo.com,YAHOO_USER,2f3f1655885efe09364846caa22fa0c4ed30c6f4769f3cb00ffb0af232f12f21
3,Abigail,Martin,abigail.martin35@hotmail.com,Liberia,Abigail Martin,hotmail.com,OTHER,eb03d267e6a7afda3971f8b8718cdbe9c2d53b3aa828008b78126df8eca2586e


Step 3: Dim Products

In [0]:
df = spark.table("databricks_de_project.silver.products")

df_dim = df.withColumn(
    "product_key",
    sha2(concat_ws("||", col("product_id"), col("product_name")), 256)
)

df_dim.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("databricks_de_project.gold.dim_products")

In [0]:
%sql
SELECT * FROM databricks_de_project.gold.dim_products;

product_id,product_name,category,price,price_category,product_key
1,Nike Samsung Galaxy,Electronics,1444.07,HIGH,0bd76cde6ac6c6b4a9bf42ec9909447954886e8e553db8e614629d20a0bcb0bf
2,HP Cookware Set,Home,1039.65,HIGH,2ddf5cc4d702beac09d9a35d584dada5c6fbbe9579dc393bd6d02c50ac9b7b1b
3,Lenovo Tennis Racket,Sports,834.93,HIGH,8d23d5ccdcb94c4429a7b2d6e29163811a5808c084a80f861a53656995e721e8
4,Puma Jeans,Clothing,950.49,HIGH,195bf003baa5e197908a22b787e00935a074b563a0f74bd4976af4ea506e38b2
5,Samsung Notebook,Books,1015.74,HIGH,2373072b106acb7e7bca09d8160455d9530c58d9539c4532068f628c7333e794
6,Puma Headphones,Electronics,915.69,HIGH,e1b4b5126bc81f87bf9259bf3e0ed6bb406a596c8f2f2426be584d344e9f270e
7,Puma iPhone,Electronics,1343.78,HIGH,6697bd6c4769a634cab2f9517527df95bd31a72f5b667a247304f7771163b235
8,Lenovo Office Chair,Home,1252.53,HIGH,855d982dfe30f91eb818037f61eeafdfe5485ef58c0cc7a661275eba837e1d9d
9,Sony Yoga Mat,Sports,891.89,HIGH,77ed74cce8c35c552321b0365c9ed04920b278ca2284c9b93e7170abd6c63adf
10,Apple Football,Sports,1044.61,HIGH,e93dd2cc7d9fe414d1fb9d73dcb1add35f6f3397973f21d24c2507dc78987207


Step 4: Fact Orders

🧠 Goal:
Join dimensions
Create analytics-ready fact table

Step 4.1: Load Tables

In [0]:
df_orders = spark.table("databricks_de_project.silver.orders_enriched")
df_customers = spark.table("databricks_de_project.gold.dim_customers")
df_products = spark.table("databricks_de_project.gold.dim_products")

Step 4.2: Join

In [0]:
df_fact = df_orders \
    .join(df_customers, "customer_id", "left") \
    .join(df_products, "product_id", "left")

Step 4.3: Select Final Columns

In [0]:
df_fact_final = df_fact.select(
    col("order_id"),
    col("customer_key"),
    col("product_key"),
    col("quantity"),
    col("order_value"),
    col("order_date"),
    col("order_year")
)

Step 4.4: Write Fact Table

In [0]:
df_fact_final.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("databricks_de_project.gold.fact_orders")

In [0]:
%sql
SELECT * FROM databricks_de_project.gold.fact_orders;

order_id,customer_key,product_key,quantity,order_value,order_date,order_year
1,95da76833d5b19d74cd2a8a6d40cb23a76a729de937edbd2e5e09789586b13a5,c3543047e54e6761659f6db5271f5efccbb4c31d155e835726d74e806f0e24be,1,429.77,2026-01-15,2026
2,7579c9531e10ea36a13b115adba7d52a2e190606c6dbcddadc12cc52abdfe026,281ffa9c13b776ea5e5435e9b55ebed16a763fcee8bff5710775bfec51e64913,4,4538.72,2026-01-27,2026
3,b78b531cd4916fe5118fab90e2060f03dd6093ffe4743c62336809bc52fb9bda,80169df8e1fd59e28f03bb4d0e9fb170bd2586789f2dba501fb047342891c9f7,4,3553.96,2026-03-13,2026
4,f66c7c6025f54f895b3f3e0cb5b79fd6ef27b66cb7eb6e105bd3c690e21962f2,7f7036c6113f2ea4dd22ea39681b446a577ea3334fb7f7051f0b430a79ce3e9c,1,1432.59,2025-11-08,2025
5,62fe37a9401943770c5b7ba46cf2f3e47f4fca8ace4f220ffebdfbb65dc11d38,44554280c29e642c876e408598d7dd6bb234407525559f8d524f5009e19aa8a2,1,519.57,2025-12-30,2025
6,33b89351907f09f762d139858fcf474db4d1a0ea5649f624c098ce3f4e7c14de,16c3a8ac72685aed625df931028c731d2a36b8783e5a2a9064fb2fbb69bd8b8a,2,1431.9,2026-02-19,2026
7,e7274f5b544916c9f81349914b66ae95bb7fc828eb7cbdbac169aeeb5c11e9e4,d0fd18e40933cf8d124f1e03b641beae66d7899e735c544960830f1f84f86c25,1,1473.71,2026-02-15,2026
8,65086e9c4e8d6bc08287cb1ce2d80ef5f891961f8158e549c63b07c44aa1a98a,d0fd18e40933cf8d124f1e03b641beae66d7899e735c544960830f1f84f86c25,1,1473.71,2026-04-13,2026
9,f5e96a5976fcef519b9bfe4563cfe2abce0ea1eb76058214ef0a46bffa0c9066,1b865a53500073f091440839d4a781e3ccaa2b029adc9d2ca80dd5fcdbeca59e,3,315.42,2026-02-26,2026
10,4db5f4102f9e9eb5fb84f54836381156419909fdb2c134897ec904bd7745f348,44554280c29e642c876e408598d7dd6bb234407525559f8d524f5009e19aa8a2,1,519.57,2025-12-03,2025


Step 5: Delta MERGE (Upsert)

Example: Upsert Fact Table

🧠 Why MERGE matters

✔ Handles updates + inserts
✔ Used in real pipelines
✔ Huge interview point

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "databricks_de_project.gold.fact_orders")

target.alias("t").merge(
    df_fact_final.alias("s"),
    "t.order_id = s.order_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

Step 6: Schema Evolution (Append + mergeSchema)

In [0]:
df_fact_final.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("databricks_de_project.gold.fact_orders")

In [0]:
%sql
SELECT * FROM databricks_de_project.gold.fact_orders;

order_id,customer_key,product_key,quantity,order_value,order_date,order_year
1,95da76833d5b19d74cd2a8a6d40cb23a76a729de937edbd2e5e09789586b13a5,c3543047e54e6761659f6db5271f5efccbb4c31d155e835726d74e806f0e24be,1,429.77,2026-01-15,2026
2,7579c9531e10ea36a13b115adba7d52a2e190606c6dbcddadc12cc52abdfe026,281ffa9c13b776ea5e5435e9b55ebed16a763fcee8bff5710775bfec51e64913,4,4538.72,2026-01-27,2026
3,b78b531cd4916fe5118fab90e2060f03dd6093ffe4743c62336809bc52fb9bda,80169df8e1fd59e28f03bb4d0e9fb170bd2586789f2dba501fb047342891c9f7,4,3553.96,2026-03-13,2026
4,f66c7c6025f54f895b3f3e0cb5b79fd6ef27b66cb7eb6e105bd3c690e21962f2,7f7036c6113f2ea4dd22ea39681b446a577ea3334fb7f7051f0b430a79ce3e9c,1,1432.59,2025-11-08,2025
5,62fe37a9401943770c5b7ba46cf2f3e47f4fca8ace4f220ffebdfbb65dc11d38,44554280c29e642c876e408598d7dd6bb234407525559f8d524f5009e19aa8a2,1,519.57,2025-12-30,2025
6,33b89351907f09f762d139858fcf474db4d1a0ea5649f624c098ce3f4e7c14de,16c3a8ac72685aed625df931028c731d2a36b8783e5a2a9064fb2fbb69bd8b8a,2,1431.9,2026-02-19,2026
7,e7274f5b544916c9f81349914b66ae95bb7fc828eb7cbdbac169aeeb5c11e9e4,d0fd18e40933cf8d124f1e03b641beae66d7899e735c544960830f1f84f86c25,1,1473.71,2026-02-15,2026
8,65086e9c4e8d6bc08287cb1ce2d80ef5f891961f8158e549c63b07c44aa1a98a,d0fd18e40933cf8d124f1e03b641beae66d7899e735c544960830f1f84f86c25,1,1473.71,2026-04-13,2026
9,f5e96a5976fcef519b9bfe4563cfe2abce0ea1eb76058214ef0a46bffa0c9066,1b865a53500073f091440839d4a781e3ccaa2b029adc9d2ca80dd5fcdbeca59e,3,315.42,2026-02-26,2026
10,4db5f4102f9e9eb5fb84f54836381156419909fdb2c134897ec904bd7745f348,44554280c29e642c876e408598d7dd6bb234407525559f8d524f5009e19aa8a2,1,519.57,2025-12-03,2025


Step 7: Analytical Queries

Top Customers

In [0]:
%sql
SELECT customer_key, SUM(order_value) AS total_spent
FROM databricks_de_project.gold.fact_orders
GROUP BY customer_key
ORDER BY total_spent DESC;

customer_key,total_spent
c79a31907d71839f4fc4d7db40464586260fe73b002d555dd441274d2ad42b44,14406.56
4db5f4102f9e9eb5fb84f54836381156419909fdb2c134897ec904bd7745f348,13842.419999999998
975038ca8bf3481d439ad5e3405209325e3584f36f3b7930af0f70063086e325,12447.96
f5e96a5976fcef519b9bfe4563cfe2abce0ea1eb76058214ef0a46bffa0c9066,12373.119999999999
0d173270bd27ece144149b1abc3336e76a0fbf443af302aaef577ac094d2f18e,11641.560000000001
7edafee874c92c04311736b1a8073d205db42b4e9a1a1fb348134994254e17dc,11237.900000000001
6716a57eb75064ce0963771d5000f4e3c19e3fb3ececdf84ac85fb4223cfc27e,10688.46
b5a9bf40ae610fb4c281382b3e0a4fcff1caae02c0cb4ae8e4a78ad6a9ef4bbd,10381.380000000001
7579c9531e10ea36a13b115adba7d52a2e190606c6dbcddadc12cc52abdfe026,9077.44
7135fb5cfe0e7fb7213a739d909e0550dd14c1ee37e61023bcb5dd58f63a92ec,8791.14


Revenue by Year

In [0]:
%sql
SELECT order_year, SUM(order_value) AS revenue
FROM databricks_de_project.gold.fact_orders
GROUP BY order_year;

order_year,revenue
2026,179187.84
2025,118066.84000000001


Top Products

In [0]:
%sql
SELECT product_key, SUM(order_value) AS revenue
FROM databricks_de_project.gold.fact_orders
GROUP BY product_key
ORDER BY revenue DESC;

product_key,revenue
3cd1969eab8f4ccdad041cfc11322e2ff89615e36b90f7c91a5f99f92d43dfe4,23903.82
281ffa9c13b776ea5e5435e9b55ebed16a763fcee8bff5710775bfec51e64913,20424.24
5749eb83aab6f0223b820db681ec2e32e61482c9737c3d6328917bc18d70c643,17532.96
80169df8e1fd59e28f03bb4d0e9fb170bd2586789f2dba501fb047342891c9f7,15992.82
d89411b57b89794fa3678b98b6daaf3b49c94c4a313e4e6672ba3e52d11edf71,11868.8
264f72a4f5f0dd86e67b4d9c0948cedc70070c8289e4ce12637636bc5c435909,11721.52
e1c527c950be155a086886c40f1ddeab68029f1ede703f587e94867f7fa2807b,9369.76
ba4d474f01468c2a741b5a744449409f647a1152e6d17e47f4c541f43132793f,8635.36
45588c432e5e325cd4832d7803473e2a08a6c292184d70c2a44e1db6e1ad2718,8264.64
dc536530899569dec1016f371fcc61ea2b0a7799f63906c844210126b983209d,7768.6


Products → SCD Type 2

In [0]:
from pyspark.sql.functions import current_timestamp, lit

df = spark.table("databricks_de_project.silver.products")

df_scd = df.withColumn("start_date", current_timestamp()) \
           .withColumn("end_date", lit(None).cast("timestamp")) \
           .withColumn("is_current", lit(True))

In [0]:
%sql
SELECT * FROM databricks_de_project.silver.products

product_id,product_name,category,price,price_category
1,Nike Samsung Galaxy,Electronics,1444.07,HIGH
2,HP Cookware Set,Home,1039.65,HIGH
3,Lenovo Tennis Racket,Sports,834.93,HIGH
4,Puma Jeans,Clothing,950.49,HIGH
5,Samsung Notebook,Books,1015.74,HIGH
6,Puma Headphones,Electronics,915.69,HIGH
7,Puma iPhone,Electronics,1343.78,HIGH
8,Lenovo Office Chair,Home,1252.53,HIGH
9,Sony Yoga Mat,Sports,891.89,HIGH
10,Apple Football,Sports,1044.61,HIGH


Fact Table Not Fully Optimized

Missing:

Partitioning ❌
Z-ordering ❌

In [0]:
df_fact_final.write.format("delta") \
    .partitionBy("order_year") \
    .mode("overwrite") \
    .saveAsTable("databricks_de_project.gold.fact_orders")

In [0]:
%sql 
OPTIMIZE databricks_de_project.gold.fact_orders
ZORDER BY (customer_key, product_key);

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 2, List(minCubeSize(107374182400), List(0, 0), List(2, 11841), 0, List(0, 0), 0, null), null, 0, 0, 2, 2, false, 0, 0, 1777197397645, 1777197398312, 8, 0, null, List(0, 0), null, 7, 7, 0, 0, null, null)"


DATA QUALITY CHECKS (Gold Layer)

Add Quality Filters

In [0]:
from pyspark.sql.functions import col

df_fact = spark.table("databricks_de_project.gold.fact_orders")

df_fact_clean = df_fact \
    .filter(col("order_value") > 0) \
    .filter(col("customer_key").isNotNull()) \
    .filter(col("product_key").isNotNull()) \
    .filter(col("order_date").isNotNull())

Overwrite Clean Table

In [0]:
df_fact_clean.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("databricks_de_project.gold.fact_orders")

validation query

In [0]:
%sql
SELECT COUNT(*) AS bad_records
FROM databricks_de_project.gold.fact_orders
WHERE order_value <= 0 OR customer_key IS NULL;

bad_records
0


AGGREGATED GOLD TABLES

Revenue Summary

In [0]:
from pyspark.sql.functions import sum as _sum

df = spark.table("databricks_de_project.gold.fact_orders")

df_revenue = df.groupBy("order_year") \
    .agg(_sum("order_value").alias("total_revenue"))

df_revenue.write.mode("overwrite") \
    .saveAsTable("databricks_de_project.gold.revenue_summary")

In [0]:
%sql
SELECT * FROM databricks_de_project.gold.revenue_summary

order_year,total_revenue
2026,89593.92
2025,59033.420000000006


Top Customers

In [0]:
df_top_customers = df.groupBy("customer_key") \
    .agg(_sum("order_value").alias("total_spent")) \
    .orderBy(col("total_spent").desc())

df_top_customers.write.mode("overwrite") \
    .saveAsTable("databricks_de_project.gold.top_customers")

In [0]:
%sql
SELECT * FROM databricks_de_project.gold.top_customers

customer_key,total_spent
c79a31907d71839f4fc4d7db40464586260fe73b002d555dd441274d2ad42b44,7203.28
4db5f4102f9e9eb5fb84f54836381156419909fdb2c134897ec904bd7745f348,6921.209999999999
975038ca8bf3481d439ad5e3405209325e3584f36f3b7930af0f70063086e325,6223.98
f5e96a5976fcef519b9bfe4563cfe2abce0ea1eb76058214ef0a46bffa0c9066,6186.5599999999995
0d173270bd27ece144149b1abc3336e76a0fbf443af302aaef577ac094d2f18e,5820.780000000001
7edafee874c92c04311736b1a8073d205db42b4e9a1a1fb348134994254e17dc,5618.950000000001
6716a57eb75064ce0963771d5000f4e3c19e3fb3ececdf84ac85fb4223cfc27e,5344.23
b5a9bf40ae610fb4c281382b3e0a4fcff1caae02c0cb4ae8e4a78ad6a9ef4bbd,5190.6900000000005
7579c9531e10ea36a13b115adba7d52a2e190606c6dbcddadc12cc52abdfe026,4538.72
7135fb5cfe0e7fb7213a739d909e0550dd14c1ee37e61023bcb5dd58f63a92ec,4395.57


Top Products

In [0]:
df_top_products = df.groupBy("product_key") \
    .agg(_sum("order_value").alias("total_revenue")) \
    .orderBy(col("total_revenue").desc())

df_top_products.write.mode("overwrite") \
    .saveAsTable("databricks_de_project.gold.top_products")

In [0]:
%sql
SELECT * FROM databricks_de_project.gold.top_products

product_key,total_revenue
3cd1969eab8f4ccdad041cfc11322e2ff89615e36b90f7c91a5f99f92d43dfe4,11951.91
281ffa9c13b776ea5e5435e9b55ebed16a763fcee8bff5710775bfec51e64913,10212.119999999999
5749eb83aab6f0223b820db681ec2e32e61482c9737c3d6328917bc18d70c643,8766.48
80169df8e1fd59e28f03bb4d0e9fb170bd2586789f2dba501fb047342891c9f7,7996.41
d89411b57b89794fa3678b98b6daaf3b49c94c4a313e4e6672ba3e52d11edf71,5934.4
264f72a4f5f0dd86e67b4d9c0948cedc70070c8289e4ce12637636bc5c435909,5860.76
e1c527c950be155a086886c40f1ddeab68029f1ede703f587e94867f7fa2807b,4684.88
ba4d474f01468c2a741b5a744449409f647a1152e6d17e47f4c541f43132793f,4317.68
45588c432e5e325cd4832d7803473e2a08a6c292184d70c2a44e1db6e1ad2718,4132.32
dc536530899569dec1016f371fcc61ea2b0a7799f63906c844210126b983209d,3884.3


Incremental Fact Load

In [0]:
from delta.tables import DeltaTable

df_new = df_fact_final  

target = DeltaTable.forName(spark, "databricks_de_project.gold.fact_orders")

target.alias("t").merge(
    df_new.alias("s"),
    "t.order_id = s.order_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

Watermark Logic

Only process new data

In [0]:
max_date = spark.sql("""
SELECT MAX(order_date) FROM databricks_de_project.gold.fact_orders
""").collect()[0][0]

df_incremental = df_orders_enriched.filter(col("order_date") > max_date)